# Project 1: Data Cleaning & Preparation 🧹
**DecodeLabs Industrial Training Kit — Batch 2026**

**Prepared by:** Ayesha Sarwar

**Goal:** Clean the raw e-commerce orders dataset by handling missing values, duplicates, and incorrect formats.

**Dataset:** `Dataset_for_Data_Analytics_-_Sheet1.csv` — 1,200 order records, 14 columns.

## 1. Setup & Initial Inspection
Load the dataset and check its shape, types, and first few rows.

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('Dataset for Data Analytics - Sheet1.csv')
print("Shape:", df.shape)
df.head()

Shape: (1200, 14)


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          1200 non-null   str    
 1   Date             1200 non-null   str    
 2   CustomerID       1200 non-null   str    
 3   Product          1200 non-null   str    
 4   Quantity         1200 non-null   int64  
 5   UnitPrice        1200 non-null   float64
 6   ShippingAddress  1200 non-null   str    
 7   PaymentMethod    1200 non-null   str    
 8   OrderStatus      1200 non-null   str    
 9   TrackingNumber   1200 non-null   str    
 10  ItemsInCart      1200 non-null   int64  
 11  CouponCode       891 non-null    str    
 12  ReferralSource   1200 non-null   str    
 13  TotalPrice       1200 non-null   float64
dtypes: float64(2), int64(2), str(10)
memory usage: 131.4 KB


## 2. Phase 1 — Strategic Imputation 🧱
"Handle the gaps. Don't just delete." We check which columns have nulls and decide a fix per column instead of blindly dropping rows.

In [6]:
missing = df.isnull().sum()
missing[missing > 0]

CouponCode    309
dtype: int64

In [7]:
# CouponCode is blank when no coupon was used — fill with explicit label
df['CouponCode'] = df['CouponCode'].fillna('NONE')

print("Remaining nulls:", df.isnull().sum().sum())

Remaining nulls: 0


## 3. Phase 2 — The Integrity Audit 🔍
"One Truth, One Record." Eliminate inflated transaction counts — unique IDs must be truly unique.

In [8]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate OrderIDs:", df['OrderID'].duplicated().sum())

Exact duplicate rows: 0
Duplicate OrderIDs: 0


In [9]:
before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset='OrderID', keep='first')
print(f"Rows removed: {before - len(df)}")
print(f"Final row count: {len(df)}")

Rows removed: 0
Final row count: 1200


## 4. Phase 3 — Speak One Language 🗣
Standardize dates to ISO 8601, trim/case text fields, and enforce numeric precision.

In [10]:
# Dates -> YYYY-MM-DD
df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
print("Unparseable dates:", df['Date'].isnull().sum())

Unparseable dates: 0


In [11]:
# Trim whitespace + standardize text casing
text_cols = ['Product', 'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'ReferralSource', 'CouponCode']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
df['CouponCode'] = df['CouponCode'].str.upper()

df[text_cols].head()

,Product,ShippingAddress,PaymentMethod,OrderStatus,ReferralSource,CouponCode
0,Monitor,928 Main St,Debit Card,Shipped,Instagram,SAVE10
1,Phone,823 Main St,Online,Shipped,Referral,SAVE10
2,Tablet,512 Main St,Credit Card,Cancelled,Email,FREESHIP
3,Chair,275 Main St,Debit Card,Returned,Facebook,SAVE10
4,Printer,668 Main St,Online,Delivered,Email,SAVE10


In [12]:
# Numeric precision (2 decimals for currency)
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce').round(2)
df['TotalPrice'] = pd.to_numeric(df['TotalPrice'], errors='coerce').round(2)

df[['UnitPrice', 'TotalPrice']].head()

,UnitPrice,TotalPrice
0,570.62,2853.10
1,151.35,302.70
2,550.68,2753.40
3,273.19,273.19
4,626.01,2504.04


**Cross-field consistency check:** Verify `TotalPrice` equals `Quantity × UnitPrice`.

In [13]:
calc = (df['Quantity'] * df['UnitPrice']).round(2)
mismatches = (calc - df['TotalPrice']).abs() > 0.01
print("Mismatches found:", mismatches.sum())

Mismatches found: 0


## 5. Validation Gate ✅
Project 2's threshold: zero duplicate IDs and zero incorrectly formatted dates.

In [14]:
import re
dup_ids = df['OrderID'].duplicated().sum()
bad_dates = (~df['Date'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$')).sum()

print(f"Duplicate IDs: {dup_ids} -> {'PASS' if dup_ids==0 else 'FAIL'}")
print(f"Bad date formats: {bad_dates} -> {'PASS' if bad_dates==0 else 'FAIL'}")
print(f"Remaining nulls: {df.isnull().sum().sum()}")

Duplicate IDs: 0 -> PASS
Bad date formats: 0 -> PASS
Remaining nulls: 0


## 6. Final Output
Save the cleaned dataset.

In [15]:
df.to_csv('cleaned_dataset.csv', index=False)
print("Saved cleaned_dataset.csv —", df.shape)

Saved cleaned_dataset.csv — (1200, 14)
